In [8]:
#!/usr/bin/env python3
"""
Integrated Order‑Flow Imbalance (OFI)
====================================

Compute the **Integrated OFI** time series exactly as defined in
*“Cross‑Impact of Order‑Flow Imbalance”*, Section 2.1.3 – 2.1.4.

Pipeline
--------
1.  Load the **normalised multi‑level OFI matrix** produced earlier
    (`multi_level_ofi.csv`).  The file must contain
       • a timestamp column called **ts_event**  
       • columns ``ofi_level_00 … ofi_level_{M‑1}``.
2.  Centre that matrix (subtract each column mean) and apply *exact* PCA
    via Singular‑Value Decomposition (SVD).
3.  Extract the first right‑singular vector ``w₁`` and L¹‑normalise it so
    that ``Σ|w₁ᵢ| = 1`` (Eq. 4).
4.  Project every timestamp’s OFI vector onto ``w₁`` to obtain a scalar
    **integrated OFI** value.
5.  Save the single‑column result to `integrated_ofi.csv`.

**Important:** This is *pure PCA*—no LASSO, sparsity, or regression tricks.

---------------------------------------------------------------------------
Author  : ChatGPT — Market‑Microstructure Feature Engineer
Version : 1.1
---------------------------------------------------------------------------

"""

from __future__ import annotations

from pathlib import Path
from typing import List

import numpy as np
import pandas as pd


# ── INPUT / OUTPUT PATHS (consistent naming across scripts) ───────────────
MULTI_OFI_CSV: Path = Path("multi_level_ofi.csv")       # ← input (normalised)
OUT_CSV:       Path = Path("integrated_ofi.csv")  # ← output (scalar series)

# Column & prefix conventions used throughout the project
COL_TIME:   str = "ts_event"       # matches earlier best‑/multi‑level scripts
OFI_PREFIX: str = "ofi_level_"     # prefix for level columns


# ── CORE FUNCTION ──────────────────────────────────────────────────────────
def compute_integrated_ofi(df_multi: pd.DataFrame) -> pd.Series:
    """
    Compute **Integrated OFI** using PCA, matching Eq. (4) of the paper.

    Parameters
    ----------
    df_multi : pandas.DataFrame
        Indexed by timestamps (UTC).  Columns must be the normalised
        multi‑level OFI features ``ofi_level_00 … ofi_level_{M‑1}``.

    Returns
    -------
    pandas.Series
        One scalar per timestamp, named ``ofi_integrated``.
    """
    # ------------------------------------------------------------------
    # 1. Extract OFI level columns in canonical order
    # ------------------------------------------------------------------
    ofi_cols: List[str] = sorted(
        [c for c in df_multi.columns if c.startswith(OFI_PREFIX)]
    )
    if not ofi_cols:
        raise ValueError(f"No columns with prefix '{OFI_PREFIX}' found.")

    X = df_multi[ofi_cols].astype(float).values  # shape = (T, M)

    # ------------------------------------------------------------------
    # 2. Centre matrix (subtract mean of each column)
    # ------------------------------------------------------------------
    X_centered = X - X.mean(axis=0, keepdims=True)

    # ------------------------------------------------------------------
    # 3. PCA via SVD (leading right‑singular vector)
    #    X_centered = U Σ Vᵀ  → first principal component = V[0]
    # ------------------------------------------------------------------
    _, _, Vt = np.linalg.svd(X_centered, full_matrices=False)
    w1 = Vt[0]                                   # shape = (M,)

    # ------------------------------------------------------------------
    # 4. L¹‑normalise so that  Σ|w₁ᵢ| = 1  (Eq. 4)
    # ------------------------------------------------------------------
    l1_norm = np.sum(np.abs(w1))
    if l1_norm == 0.0:
        raise ValueError("Leading principal component has zero L¹‑norm.")
    w1_norm = w1 / l1_norm

    # ------------------------------------------------------------------
    # 5. Project each timestamp’s OFI vector onto w₁
    # ------------------------------------------------------------------
    integrated_values = X @ w1_norm              # shape = (T,)

    return pd.Series(
        integrated_values,
        index=df_multi.index,
        name="ofi_integrated",
        dtype=float,
    )


# ── MAIN (example usage) ──────────────────────────────────────────────────
if __name__ == "__main__":
    # 1. Load the multi‑level OFI matrix
    df_multi = (
        pd.read_csv(
            MULTI_OFI_CSV,
            parse_dates=[COL_TIME],
            index_col=COL_TIME,
        )
        .sort_index()            # ensure chronological order
    )

    # 2. Compute Integrated OFI
    ofi_integrated = compute_integrated_ofi(df_multi)

    # 3. CSV & show quick preview
    ofi_integrated.to_csv(OUT_CSV, header=True)
    print(f"Integrated OFI written to {OUT_CSV}  ({len(ofi_integrated)} rows)")
    print(ofi_integrated.head())


Integrated OFI written to integrated_ofi.csv  (71 rows)
ts_event
2024-10-21 11:55:00+00:00    0.711061
2024-10-21 11:56:00+00:00    0.738289
2024-10-21 11:57:00+00:00    1.050139
2024-10-21 11:58:00+00:00    0.370352
2024-10-21 11:59:00+00:00    1.899927
Name: ofi_integrated, dtype: float64
